# Decomposition: 15 retrained checkpoints and official Chronos

Companion to `appendix_decomposition.ipynb`; the original notebook is unchanged.

The figures compare the same real signal, the selected retrained checkpoint and the
official `amazon/chronos-bolt-tiny` (16,16). Both outputs receive the same context.
Component membership is selected independently from each output's Hann spectrum.

The final section measures frequency recovery in **100 lock and 100 control trials
per checkpoint and family**, for all 15 frozen geometries plus the official model.
TSMixup and KernelSynth have separate 16-row tables. The two 16,16 checkpoints receive
identical inputs. Figure settings and benchmark settings are independent.

Recovery is the presence of a selected peak within **inclusive ±1 Hz** of the injected
frequency in the direct **64-point q50 forecast**. Zero padding evaluates a 1 Hz grid;
native FFT spacing remains 8 Hz. This is a descriptive recovery measure, not causal
evidence of aliasing or proof that a background component was absent before injection.


## 0, Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Where the repository is. Locally the notebook sits inside it and nothing has to be set; on
# Colab the working directory is /content, so the search widens to the usual places and, failing
# those, the repository is cloned. Setting REPO_DIR (or the PATCHALIASING_REPO environment
# variable) to the checkout skips the search entirely.
REPO_DIR = None                    # e.g. "/content/drive/MyDrive/patchAliasing"
REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
CLONE_IF_MISSING = True            # False to fail with a message instead of cloning

MARKER = Path("chronos") / "bayesian" / "probe_lib.py"


def _ok(p) -> bool:
    return p is not None and (Path(p) / MARKER).exists()


def find_repo() -> Path:
    """Locate the checkout: an explicit setting, then the parents, then the usual Colab places."""
    explicit = REPO_DIR or os.environ.get("PATCHALIASING_REPO")
    if explicit:
        if _ok(explicit):
            return Path(explicit).resolve()
        raise FileNotFoundError(f"REPO_DIR is set to {explicit}, but {MARKER} is not under it")

    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:                       # the notebook inside the checkout
        if _ok(cand):
            return cand

    roots = [here, Path("/content"), Path("/content/drive/MyDrive"),
             Path("/content/drive/MyDrive/Colab Notebooks"), Path.home()]
    for root in roots:                                       # a checkout beside the notebook
        if not root.exists():
            continue
        for cand in [root, *(d for d in root.iterdir() if d.is_dir())]:
            if _ok(cand):
                return cand.resolve()

    if not CLONE_IF_MISSING:
        raise FileNotFoundError(
            f"{MARKER} not found. Set REPO_DIR to the checkout, or allow CLONE_IF_MISSING.")

    target = here / "patchAliasing"                          # last resort: fetch it
    if not (target / ".git").exists():
        print(f"cloning {REPO_URL} -> {target}")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    if not _ok(target):
        raise FileNotFoundError(f"{MARKER} missing from {target}")
    return target.resolve()


REPO  = find_repo()
BAYES = REPO / "chronos" / "bayesian"
sys.path.insert(0, str(BAYES))
sys.modules.pop("probe_lib", None)          # so a git pull is picked up without a kernel restart
print("repository:", REPO)

In [ ]:
# Dependencies.  A local machine that has already run the Bayesian notebooks has all of these;
# a fresh Colab runtime has numpy, pandas, matplotlib, scikit-learn and torch, but not the
# Chronos pipeline classes, which come from `chronos-forecasting`.
import importlib.util, subprocess, sys

for module, package in (("torch", "torch"), ("chronos", "chronos-forecasting")):
    if importlib.util.find_spec(module) is None:
        print(f"installing {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import chronos
# The repository has its own top-level `chronos/` directory.  It shadows the library whenever the
# working directory is the repository root, and the error that follows is confusing, so it is
# caught here instead.
if not hasattr(chronos, "BaseChronosPipeline"):
    raise ImportError(
        f"`chronos` resolved to {getattr(chronos, '__file__', '?')}, which is the repository's own "
        "folder rather than the chronos-forecasting library. Run this notebook from a working "
        "directory that is not the repository root.")
print("chronos:", chronos.__file__)

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")
import probe_lib as pl

SEED = [11, 17, 2, 33, 46, 59, 60, 74, 85, 9, 42]
np.random.seed(SEED)
FS, CTX, PRED, BAND = pl.FS, pl.CTX, pl.PRED, pl.BAND

from matplotlib.patches import ConnectionPatch

# ------------------------------------------------------------------------------------- #
#  CONFIGURATION
# ------------------------------------------------------------------------------------- #
FAMILY = "kernelsynth"          # "tsmixup" or "kernelsynth": the signal family

# Which model reconstructs the signal.
#   "retrained": the sweep checkpoint of the geometry below, the one the report analyses
#   "published": the original Chronos-Bolt-Tiny released by Amazon, untouched by this project
# The published tiny model is itself P = S = 16, so the two are the same geometry trained by
# different people on different data, which is what makes them worth putting side by side.
MODEL_SOURCE = "retrained"
PUBLISHED_ID = "amazon/chronos-bolt-tiny"
REF_P, REF_S = 16, 12       # used when MODEL_SOURCE is "retrained"; the published one carries
                            # its own, and the geometry is read back off whichever is loaded

# Light TSMixup draw. K is the number of components the page shows: raise it for a richer
# mixture, at the cost of one more row of panels. The generator itself picks k ~ U{1, K}, which
# K_RANDOM reproduces; fixed K is the default so the two families give pages of equal height.
K_COMPONENTS = 4
K_RANDOM     = True         # True: k ~ U{1, K}, as the generator does
ALPHA_DIR    = 1.5          # Dirichlet concentration
MIN_SEP_DRAW = 10.0         # Hz between two drawn components
POOL_FMAX    = 250.0        # ceiling applied to the draw made FOR THESE FIGURES only. probe_lib's
                            # pool spans the whole analysed band, 2-250 Hz, as Appendix C states;
                            # the cap here keeps the drawn components in the range where the model
                            # still returns something, so the panels show a response rather than a
                            # flat line. It is a display choice and not a property of the pool.
                            # supplies. A checkout whose probe_lib predates TSMIXUP_POOL_FMAX
                            # still builds the pool to the top of the band, and this keeps the
                            # notebook's pool the intended one either way. None = leave as found
DRAW_FMIN    = None         # narrow the range a single draw may use, inside the pool above.
DRAW_FMAX    = None         # None uses the pool's own range

# How many components each side shows. A Light TSMixup draw holds at most K of them and they are
# all displayed; the prediction is under no such constraint -- it may hold lines the input never
# had -- so it gets its own, larger count. For KernelSynth the input has no components to speak of
# and its strongest lines stand in for them.
N_PEAKS_REAL = 6            # KernelSynth only: lines read off the real signal. A KernelSynth
                            # draw is broadband and carries no drawn frequencies to name, so its
                            # real side is read as lines, as the prediction's is
N_PEAKS_PRED = 8            # lines read off the prediction, on pages B and C
SPECTRUM_BIN_HZ = 1.0       # spacing of the frequency grid the transforms are evaluated on.
                            # Reached by zero padding, which interpolates the spectrum: it places
                            # a peak far more precisely than the raw bin spacing would, but it
                            # does NOT let two nearby lines be told apart -- that stays fs/N,
                            # printed beside it as the resolving power
PEAK_MIN_SEP = 6.0          # Hz between two peaks, so one lobe is not counted twice
PEAK_MIN_REL = 0.05         # and a line below this fraction of the strongest one is not a line

# How much of each signal the least-squares fit sees. The rollout is only faithful near its
# start -- it is fed its own output and drifts -- so the prediction's amplitude and phase are read
# off its first samples rather than off the whole trace. Lines are still *found* on the full
# trace, where the spectrum has resolution; only the fit is windowed.
PRED_CUT        = None      # samples of the prediction kept; None keeps it whole. Everything
                            # downstream -- plots, peak search, fits -- sees only what is kept
MATCH_TIME_SCALE = True     # panels drawn to the same milliseconds-per-inch, so a shorter
                            # prediction comes out as a narrower panel. False makes them equal
REAL_SEGMENT = "future"     # which stretch of the real signal is shown and fitted beside it:
                            #   "future"       the true continuation, the same instants the
                            #                  prediction covers -- the like-for-like comparison
                            #   "context_tail" the last samples the model was given
                            #   "full"         the whole draw
FIT_WINDOW_PRED = 64        # samples used for the fit, clamped to what PRED_CUT kept; None = all
FIT_WINDOW_REAL = None      # the same for the real signal
MIN_CYCLES_WARN = 1.5       # a fit spanning fewer cycles than this is flagged on the panel
SPECTRAL_CUT_REL = 0.35     # keep a prediction line only when the Hann-spectrum magnitude
                            # plotted in Figure 4 reaches this fraction of its strongest peak
SPECTRAL_CUT_ABS = None     # optional absolute floor in the same plotted magnitude units;
                            # when set, it overrides SPECTRAL_CUT_REL
COMPONENT_WINDOW = "signal" # time axis of the component panels:
                            #   "signal": the same absolute window as the signal it came from,
                            #             so every panel of the page shares one time axis
                            #   "cycles": a few cycles from zero, easier to read one waveform
COMPONENT_CYCLES = 4        # how many, when COMPONENT_WINDOW is "cycles"
AMP_SCALE       = "real"    # vertical scale of the component panels of figure 3:
                            #   "real": one scale for the whole page, set by the strongest
                            #           component of the real signal, so a weak prediction reads
                            #           as weak instead of being blown up by autoscaling
                            #   "own":  each panel scaled to what it holds
JOINT_FIT       = True      # fit all the frequencies of a page together rather than one at a
                            # time. Over a short window two frequencies inside one bin are nearly
                            # the same basis vector, and separate fits then report the same energy
                            # twice; a joint fit shares it out instead, and says when it cannot

OUT_MODE  = "horizon"       # "horizon": the model's own forecast, PRED samples, which is what a
                            #            deployment actually gets and is far shorter than the context
                            # "rollout": feed the median forecast back until GEN_LEN samples. Finer
                            #            spectra, but the trace is partly the model reading itself
GEN_LEN   = 512             # rollout length, so the output spectrum has 1 Hz bins
BATCH     = 64
DRAW_SEED = SEED            # one seed, or a vector such as [42, 50, 77]
SAVE_PDF  = "n"            # Y/n: also save each figure as PDF; default n writes PNG only
INJECT_TONE = None          # Hz, or None: an arbitrary probe tone added to the background

# A component planted on one of the geometry's own lock frequencies, before the signal is given
# to the model. This is the deliberate version of the question the report asks: the input then
# certainly carries energy at a predicted site, and the figures show what comes back.
INJECT_LOCK       = False   # True to plant it
INJECT_LOCK_HZ    = None    # a specific lock frequency, or None to draw one at random
INJECT_LOCK_REL   = 1.25    # its amplitude, as a multiple of the strongest existing component
INJECT_LOCK_IN_POOL = True  # draw only from locks the source pool could also have supplied
INJECT_LOCK_PHASE = "random"   # "random", or a phase in radians

USE_STUB_FORECASTER = False # True only to check the layout without the checkpoints

def normalise_seeds(value):
    """Return the configured seed or seed vector as a non-empty list of distinct integers."""
    if isinstance(value, (str, bytes)):
        raise TypeError("DRAW_SEED must be an integer or an iterable of integers")
    values = [value] if np.isscalar(value) else list(value)
    if not values:
        raise ValueError("DRAW_SEED must contain at least one seed")
    return list(dict.fromkeys(int(seed) for seed in values))


def pdf_requested():
    """Interpret the user-facing Y/n PDF flag, rejecting a mistyped value."""
    flag = str(SAVE_PDF).strip().lower()
    if flag not in {"y", "n"}:
        raise ValueError("SAVE_PDF must be 'Y' or 'n'")
    return flag == "y"


DRAW_SEEDS = normalise_seeds(DRAW_SEED)
ACTIVE_SEED = DRAW_SEEDS[0]
PRIMARY_FAMILY = FAMILY
import comparison_lib as comparison
import comparison_figures as comparison_plot
import torch
from dataclasses import replace

COMPARISON_MODE = os.environ.get("PATCHALIASING_COMPARISON_MODE", "full")
if COMPARISON_MODE not in {"full", "smoke"}:
    raise ValueError("COMPARISON_MODE must be full or smoke")
torch.set_num_threads(8)
RUN_RECOVERY_TABLES = True
COMPARISON_ROOT = BAYES / "_run" / "comparison"
RECOVERY_CONFIG = comparison.RecoveryConfig()  # independent of all figure controls above
COMPARISON_MODELS = comparison.MODELS
if COMPARISON_MODE == "smoke":
    RECOVERY_CONFIG = replace(RECOVERY_CONFIG, mode="smoke", n_backgrounds=4)
    COMPARISON_MODELS = (comparison.MODELS[0], comparison.MODELS[4])
    DRAW_SEED = [42]
    print("SMOKE ONLY: reduced sample and checkpoint set; not a full comparison")
OUT = BAYES / "_run" / "appendixE_comparison" / COMPARISON_MODE
DRAW_SEEDS = normalise_seeds(DRAW_SEED)
ACTIVE_SEED = DRAW_SEEDS[0]
(OUT / "figures").mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(ACTIVE_SEED)

NAME = {"tsmixup": "Light TSMixup", "kernelsynth": "KernelSynth"}[PRIMARY_FAMILY]
INJECTED_LOCK = None        # the planted component, set in section 1 when INJECT_LOCK is on


def run_suffix():
    """The tag every file of this run carries: family, model, and the planted component.

    It is a function and not a constant, and it reads the settings as they stand when it is
    called. The geometry it names is therefore the one actually loaded in section 2 rather than
    the one requested here, a run on the published checkpoint always says `published`, and a run
    that plants a component says where. `seeded_name` then appends the active seed, so runs with
    different draws cannot overwrite one another.
    """
    if USE_STUB_FORECASTER:
        src = "stub"
    elif MODEL_SOURCE == "published":
        src = "published"
    else:
        src = f"p{REF_P}-s{REF_S}"
    tag = f"{FAMILY}_{src}"
    if INJECTED_LOCK:
        tag += "_lock" + f"{INJECTED_LOCK:g}".replace(".", "p")
    return tag


def seeded_name(stem, extension):
    """Append the active seed as the final filename token before the extension."""
    return f"{stem}_seed{ACTIVE_SEED}{extension}"


print(f"family={NAME}  model={MODEL_SOURCE}  K={K_COMPONENTS}"
      f"{' (random)' if K_RANDOM else ''}  output='{OUT_MODE}'  seeds={DRAW_SEEDS}")
print(f"fit window: prediction {FIT_WINDOW_PRED or 'all'} samples, "
      f"real {FIT_WINDOW_REAL or 'all'} samples")
print(f"components shown: real {'K = %d' % K_COMPONENTS if FAMILY == 'tsmixup' else 'up to %d lines' % N_PEAKS_REAL}, "
      f"prediction up to {N_PEAKS_PRED} lines")
print(f"figures -> {OUT / 'figures'}  PDF={'Y' if pdf_requested() else 'n'}")
print(f"files for the first draw end in  _seed{ACTIVE_SEED}.*  "
      f"(the family/model tag is settled by the checkpoint that actually loads)")

## 1, The signal

In [ ]:
# The source pool of Light TSMixup. `probe_lib.tsmixup_pool` is used when the checkout has it;
# a checkout that predates it gets the same construction rebuilt here, from primitives both
# versions carry. The four groups are those of Appendix B: every integer of the band, the
# non-integer members of F_lock, the control frequencies of every candidate, and 150 further
# non-integer frequencies held clear of both.
POOL_N_FREE, POOL_GUARD = 150, 0.2


def _pool_rebuilt(n_free=POOL_N_FREE, guard=POOL_GUARD, models=None):
    models = models or pl.MODELS
    lo = pl.BAND[0]
    # the pool's own ceiling when the checkout defines one, the analysed band otherwise
    hi = min(pl.BAND[1], float(getattr(pl, "TSMIXUP_POOL_FMAX", pl.BAND[1])),
             pl.BAND[1] if POOL_FMAX is None else float(POOL_FMAX))
    lock = set()
    for P, S in models:
        lock |= {round(f, 9) for f in pl.f_lock(P, S, fmax=hi, fmin=lo)}
    lock_sorted = sorted(lock)
    integers = [float(f) for f in range(int(np.ceil(lo)), int(np.floor(hi)) + 1)]
    non_integer_locks = [f for f in lock_sorted if abs(f - round(f)) > 1e-9]
    controls = set()
    for P, S in models:
        for fk in pl.f_lock(P, S, fmax=hi, fmin=lo):
            d = pl.control_offset(P, S, fk)
            if not np.isfinite(d):
                continue
            for f in (fk - d, fk + d):
                if lo <= f <= hi:
                    controls.add(round(f, 9))

    def clear(f):
        return (min(abs(f - l) for l in lock_sorted) >= guard) and (abs(f - round(f)) >= guard)

    free = []
    for f0 in np.linspace(lo + 0.5, hi - 0.5, n_free):
        f, j = float(f0), 0
        while not clear(f) and j < 40:                     # nudge until it is clear of both grids
            j += 1
            f = float(f0) + 0.05 * ((j + 1) // 2) * (1 if j % 2 else -1)
        free.append(round(f, 4))
    return sorted(set(integers) | {round(f, 9) for f in non_integer_locks} | controls | set(free))


_POOL_CACHE = {}


def source_pool(verbose=True):
    """The Light TSMixup pool, from probe_lib when it has one, rebuilt here otherwise."""
    if "pool" not in _POOL_CACHE:
        fn = getattr(pl, "tsmixup_pool", None)
        if fn is None:
            print("note: this checkout of probe_lib has no tsmixup_pool; the pool is rebuilt "
                  "in-notebook. Push the updated probe_lib.py to keep the two from drifting.")
            _POOL_CACHE["pool"] = _pool_rebuilt()
            _POOL_CACHE["source"] = "rebuilt in-notebook"
        else:
            _POOL_CACHE["pool"] = fn(models=list(comparison.FROZEN15))
            _POOL_CACHE["source"] = "probe_lib.tsmixup_pool"
    pool = _POOL_CACHE["pool"]
    if POOL_FMAX is not None:                    # hold the ceiling whatever the checkout gave
        pool = [f for f in pool if f <= float(POOL_FMAX)]
    if verbose and not _POOL_CACHE.get("announced"):
        _POOL_CACHE["announced"] = True
        lib_cap = getattr(pl, "TSMIXUP_POOL_FMAX", None)
        where = ("probe_lib defines the cap" if lib_cap is not None
                 else "this checkout's probe_lib has no cap; applied here")
        print(f"pool: {len(pool)} frequencies from {min(pool):g} to {max(pool):g} Hz "
              f"({_POOL_CACHE['source']}; {where})")
    return pool


def light_tsmixup_draw(rng, k=None, alpha=None, length=pl.CANON_LEN, min_sep=None, pool=None):
    """One Light TSMixup realisation, with the frequencies and weights of the draw returned.

    Algorithm 1 of the appendix: one sinusoid per drawn pool frequency, each divided by its mean
    absolute value, combined under symmetric Dirichlet weights, normalised to unit variance.

    The settings are read from the configuration cell at call time, not bound as defaults: a
    default would freeze the value this cell last ran with, so editing K and re-running only the
    configuration would silently keep the old number of components.
    """
    k = K_COMPONENTS if k is None else int(k)
    alpha = ALPHA_DIR if alpha is None else float(alpha)
    min_sep = MIN_SEP_DRAW if min_sep is None else float(min_sep)
    pool = np.asarray(source_pool() if pool is None else pool, dtype=float)
    lo = min(pool) if DRAW_FMIN is None else float(DRAW_FMIN)
    hi = max(pool) if DRAW_FMAX is None else float(DRAW_FMAX)
    full = len(pool)
    pool = pool[(pool >= lo) & (pool <= hi)]
    if len(pool) < full:
        print(f"draw restricted to [{lo:g}, {hi:g}] Hz: {len(pool)} of {full} pool frequencies")
    if len(pool) < k:
        raise ValueError(f"only {len(pool)} frequencies in [{lo:g}, {hi:g}] Hz, K = {k}")
    if K_RANDOM:
        k = int(rng.integers(1, k + 1))              # the generator's own k ~ U{1, K}
    freqs = []
    while len(freqs) < k:                       # redraw a frequency that would overlap another
        f = float(rng.choice(pool))
        if all(abs(f - g) >= min_sep for g in freqs):
            freqs.append(f)
    comps  = [np.asarray(pl.make_tone(f, 0.0, length, 1.0), dtype=float) for f in freqs]
    scaled = [c / np.mean(np.abs(c)) for c in comps]
    w = rng.dirichlet(np.full(k, alpha))
    x = np.sum([wi * si for wi, si in zip(w, scaled)], axis=0)
    return (x / x.std()).astype(np.float32), sorted(freqs), w[np.argsort(freqs)]


def spectrum(x, fs=FS, window=True, bin_hz=None):
    """One-sided magnitude spectrum on a grid of `bin_hz`, scaled so a tone of amplitude A reads A.

    The grid is reached by zero padding. That interpolates between the true bins and lets a peak
    be placed to a fraction of them; it does not add resolving power, which stays fs/N.
    """
    bin_hz = SPECTRUM_BIN_HZ if bin_hz is None else float(bin_hz)
    y = np.asarray(x, float); y = y - y.mean()
    w = np.hanning(len(y)) if window else np.ones(len(y))
    nfft = max(len(y), int(round(fs / max(bin_hz, 1e-9))))
    mag = np.abs(np.fft.rfft(y * w, n=nfft)) * 2.0 / np.sum(w)
    return np.fft.rfftfreq(nfft, d=1 / fs), mag


def locks_of(P, S):
    """The predicted sites of one geometry: F_lock = {k fs/P} union {c fs/S}."""
    return sorted({round(f, 3) for f in pl.patch_nulls(P)} |
                  {round(f, 3) for f in pl.stride_locks(S)})


def is_lock(f, tol=1.0):
    """Whether `f` sits on the comb of the geometry currently selected."""
    return any(abs(f - l) <= tol for l in locks_of(REF_P, REF_S))


def pool_limit():
    """The highest frequency the input could possibly contain, for this family and this draw."""
    if FAMILY != "tsmixup":
        return None
    cap = float(getattr(pl, "TSMIXUP_POOL_FMAX", BAND[1]))
    if POOL_FMAX is not None:
        cap = min(cap, float(POOL_FMAX))
    if DRAW_FMAX is not None:
        cap = min(cap, float(DRAW_FMAX))
    return min(cap, BAND[1])


def above_pool(f):
    """Whether `f` is above anything the generator could have put in: the model's own content."""
    cap = pool_limit()
    return cap is not None and f > cap + 1e-6


def top_peaks(x, n_peaks=None, min_sep=None, band=BAND, fs=FS, min_rel=None):
    """The strongest spectral lines of `x` inside `band`.

    Candidates must be local maxima of the plotted spectrum before the analysis-band filter is
    applied; a band edge therefore cannot become a peak merely because its outside neighbour was
    cropped away. At most `n_peaks` are returned,
    no two closer than `min_sep`, and none weaker than `min_rel` of the strongest: a broad lobe
    therefore contributes its summit, not several high points on the same shoulder.
    """
    n_peaks = N_PEAKS_PRED if n_peaks is None else int(n_peaks)
    # never below the resolving power: two "peaks" closer than fs/N are one line seen twice,
    # whatever grid the transform is evaluated on
    min_sep = max(PEAK_MIN_SEP, fs / len(np.asarray(x))) if min_sep is None else float(min_sep)
    min_rel = PEAK_MIN_REL if min_rel is None else float(min_rel)
    fr, mag = spectrum(x, fs=fs)
    in_band = (fr >= band[0]) & (fr <= band[1])
    band_mag = mag[in_band]
    if not len(band_mag) or band_mag.max() <= 0:
        return []
    floor = min_rel * band_mag.max()
    local = np.zeros(len(mag), dtype=bool)
    if len(mag) == 1:
        local[0] = True
    else:
        local[0] = mag[0] >= mag[1]
        local[-1] = mag[-1] > mag[-2]
        local[1:-1] = (mag[1:-1] > mag[:-2]) & (mag[1:-1] >= mag[2:])
    candidates = np.flatnonzero(local & in_band & (mag >= floor))
    picked = []
    for i in candidates[np.argsort(mag[candidates])[::-1]]:
        f = float(fr[i])
        if all(abs(f - p) >= min_sep for p in picked):
            picked.append(f)
        if len(picked) >= n_peaks:
            break
    return sorted(picked)


def kernelsynth_draw(seed, length=pl.CANON_LEN):
    """One KernelSynth realisation, with the kernels of the draw recorded.

    `probe_lib.background` does exactly this and hands back the array alone; the generator is
    called here with the same parameters and the same seed, so the signal is the one probe_lib
    would have produced and `last_kernels` can be read off it. `signalGenerator` is importable
    because probe_lib puts the generator directory on the path when it is imported.
    """
    import signalGenerator as sg
    tmp = BAYES / "_gen_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    params = {"J": 5, "l_syn": int(length), "fs": FS, "jitter": 1e-4, "P": 16}
    gen = sg.runKernelSynth(params, int(seed), tmp)
    x = np.asarray(gen.generate(), float).ravel()[:int(length)]
    sd = x.std()
    x = (x / sd) if sd > 1e-8 else x                  # unit variance, as probe_lib does
    return x.astype(np.float32), list(getattr(gen, "last_kernels", [])), dict(gen.getParameters())


def amp_of(x, f):
    """Least-squares amplitude of `x` at `f`, before the decomposition cell defines its own."""
    y = np.asarray(x, float)
    return float(pl.fit_amp_phase(y, np.arange(len(y)) / FS, f)[0])


def plant_lock_component(x, rng):
    """Add a component on one of the geometry's lock frequencies, and return it with the choice.

    The amplitude is set relative to the strongest component the signal already has, so the
    planted tone is the dominant one by construction and cannot be missed for want of energy.
    The result is renormalised to unit variance, as every background here is, which scales the
    whole mixture and leaves the ratio between components untouched.
    """
    locks = [f for f in locks_of(REF_P, REF_S) if BAND[0] <= f <= BAND[1]]
    cap = pool_limit()
    if INJECT_LOCK_IN_POOL and cap is not None:
        locks = [f for f in locks if f <= cap] or locks
    if not locks:
        raise ValueError(f"p{REF_P}-s{REF_S} has no lock frequency in {BAND}")
    f = float(INJECT_LOCK_HZ) if INJECT_LOCK_HZ is not None else float(rng.choice(locks))
    base = max([amp_of(x, g) for g in REF_FREQS] + [1e-9])
    amp = INJECT_LOCK_REL * base
    phase = (float(rng.uniform(0, 2 * np.pi)) if INJECT_LOCK_PHASE == "random"
             else float(INJECT_LOCK_PHASE))
    y = np.asarray(x, float) + np.asarray(pl.make_tone(f, phase, len(x), amp), float)
    return (y / y.std()).astype(np.float32), f, amp, phase


def build_signal(family=None, seed=None):
    """Draw one realisation and bind everything the later sections read off it.

    This section is a function rather than a run of statements so section 6 can run every
    requested seed for both generators. What is bound here is exactly what figures 3 and 4 read.
    """
    global FAMILY, NAME, RNG, ACTIVE_SEED, SIGNAL, CONTEXT, REF_FREQS, REF_SOURCE, WEIGHTS
    global KERNELS, KS_PARAMS, INJECTED_LOCK
    if family is not None:
        FAMILY = str(family)
    if seed is not None:
        ACTIVE_SEED = int(seed)
    NAME = {"tsmixup": "Light TSMixup", "kernelsynth": "KernelSynth"}[FAMILY]
    np.random.seed(ACTIVE_SEED)
    RNG = np.random.default_rng(ACTIVE_SEED)
    KERNELS, KS_PARAMS, WEIGHTS, INJECTED_LOCK = [], {}, None, None
    print(f"--- {NAME}, seed {ACTIVE_SEED} " + "-" * 40)

    if FAMILY == "tsmixup":
        SIGNAL, REF_FREQS, WEIGHTS = light_tsmixup_draw(RNG)
        REF_SOURCE = "the frequencies drawn by the generator"
        print(f"drew {len(REF_FREQS)} components (K = {K_COMPONENTS}"
              f"{', random' if K_RANDOM else ''})")
        print("drawn components [Hz]:", [round(f, 3) for f in REF_FREQS])
        print("mixing weights      :", np.round(WEIGHTS, 3))
        hi_used = pool_limit() or max(source_pool(verbose=False))
        assert max(REF_FREQS) <= hi_used + 1e-6, (
            f"drew {max(REF_FREQS):g} Hz above the {hi_used:g} Hz limit: the pool or the cell "
            "defining light_tsmixup_draw is stale, re-run it")
        if not K_RANDOM:
            assert len(REF_FREQS) == K_COMPONENTS, (
                f"{len(REF_FREQS)} components for K = {K_COMPONENTS}: the cell defining "
                "light_tsmixup_draw is stale, re-run it")
    else:
        SIGNAL, KERNELS, KS_PARAMS = kernelsynth_draw(ACTIVE_SEED)
        REF_FREQS, WEIGHTS = top_peaks(SIGNAL, n_peaks=N_PEAKS_REAL), None
        REF_SOURCE = f"the {len(REF_FREQS)} strongest lines of the generated signal"
        print("kernels drawn      :", " ".join(KERNELS))
        print("generator settings :", {k: v for k, v in KS_PARAMS.items() if k != "inject"})
        print("strongest lines [Hz]:", [round(f, 2) for f in REF_FREQS])

    if INJECT_TONE is not None:
        SIGNAL = (SIGNAL + pl.make_tone(INJECT_TONE, 0.0, len(SIGNAL), pl.TONE_SNR)).astype(np.float32)
        print(f"probe tone injected at {INJECT_TONE} Hz")

    if INJECT_LOCK:
        SIGNAL, INJECTED_LOCK, _amp, _ph = plant_lock_component(SIGNAL, RNG)
        REF_FREQS = sorted(set(list(REF_FREQS) + [INJECTED_LOCK]))
        print(f"planted a component at {INJECTED_LOCK:g} Hz, a lock of p{REF_P}-s{REF_S}: "
              f"amplitude {INJECT_LOCK_REL:g}x the strongest existing component, "
              f"phase {_ph:.2f} rad")
        print("   after renormalising, the components are: "
              + ", ".join(f"{f:g} Hz A={amp_of(SIGNAL, f):.3f}"
                          + ("  <- planted" if f == INJECTED_LOCK else "") for f in REF_FREQS))

    CONTEXT = SIGNAL[:CTX]
    print(f"signal {len(SIGNAL)} samples, context {len(CONTEXT)}, std {SIGNAL.std():.3f}")
    return SIGNAL


build_signal(PRIMARY_FAMILY, DRAW_SEEDS[0])

In [ ]:
# What produced this signal, in full: a figure is only as good as the draw behind it, and both
# families are reproducible from what is printed here plus ACTIVE_SEED.


def describe_draw(show=True):
    """The parameters of the draw currently in memory, printed, tabulated and archived."""
    global PARAMS, SETTINGS
    if FAMILY == "tsmixup":
        drawn = [f for f in REF_FREQS if f != INJECTED_LOCK]
        w_of = dict(zip(sorted(drawn), [WEIGHTS[i] for i in np.argsort(np.argsort(drawn))]))
        PARAMS = pd.DataFrame({
            "component": np.arange(1, len(REF_FREQS) + 1),
            "frequency_hz": REF_FREQS,
            "weight": [w_of.get(f, np.nan) for f in REF_FREQS],
            "planted": [f == INJECTED_LOCK for f in REF_FREQS],
            "in_F_lock": [is_lock(f) for f in REF_FREQS],
            "cycles_per_patch": [f * REF_P / FS for f in REF_FREQS],
        })
        SETTINGS = {"generator": "Light TSMixup", "K": K_COMPONENTS, "k_random": K_RANDOM,
                    "planted_lock_hz": INJECTED_LOCK,
                    "alpha": ALPHA_DIR, "pool_size": len(source_pool()),
                    "min_separation_hz": MIN_SEP_DRAW,
                    "draw_range_hz": (BAND[0] if DRAW_FMIN is None else DRAW_FMIN,
                                      BAND[1] if DRAW_FMAX is None else DRAW_FMAX),
                    "length": len(SIGNAL),
                    "fs_hz": FS, "seed": ACTIVE_SEED}
    else:
        PARAMS = pd.DataFrame({"kernel": KERNELS})
        SETTINGS = {"generator": "KernelSynth", **{k: v for k, v in KS_PARAMS.items()
                                                   if k != "inject"},
                    "n_kernels_drawn": len(KERNELS), "planted_lock_hz": INJECTED_LOCK,
                    "seed": ACTIVE_SEED}

    print("settings:")
    for k, v in SETTINGS.items():
        print(f"    {k:>18} = {v}")
    PARAMS.to_csv(OUT / seeded_name(f"generator_params_{run_suffix()}", ".csv"), index=False)
    if show:
        display(PARAMS.round(4))
    return PARAMS


describe_draw()

## 2, What the model returns

In [ ]:
class StubProbe:
    """A stand-in for `pl.Probe` that needs no checkpoint: layout checks only.

    It returns a smoothed continuation of the context, which is not a forecast and must never be
    read as one. Every figure produced while `USE_STUB_FORECASTER` is true carries a stamp.
    """
    def __init__(self, P, S):
        self.P, self.S = P, S
        self.tag, self.label = pl.model_tag(P, S), f"stub p{P}-s{S}"
        self.stages = ["output_head"]

    def forecast(self, contexts):
        c = np.asarray(contexts, dtype=np.float32)
        k = np.ones(9) / 9.0
        sm = np.stack([np.convolve(row, k, mode="same") for row in c])
        return np.repeat(sm[:, -1:], PRED, axis=1) * 0.6 + sm[:, -PRED:] * 0.4

    def capture_reg(self, contexts, pipe=None):
        c = np.asarray(contexts, dtype=np.float32)
        return {"output_head": np.stack([c[:, :16], c[:, -16:]], axis=1).reshape(len(c), -1)}

    def close(self):
        pass


def open_probe(P, S, batch_size=64):
    """The real probe, or the stub when the checkpoints are not available."""
    return StubProbe(P, S) if USE_STUB_FORECASTER else pl.Probe(P, S, batch_size=batch_size)


def stamp_stub(fig):
    if USE_STUB_FORECASTER:
        fig.text(0.5, 0.5, "STUB FORECASTER\nNOT A MEASUREMENT", fontsize=42, color="red",
                 alpha=0.16, ha="center", va="center", rotation=30, zorder=99)

In [ ]:
def chronos_generate(probe, context, mode=OUT_MODE, gen_len=GEN_LEN):
    """The model's output for one context: the raw horizon, or the fed-back rollout.

    The rollout is the procedure the reconstruction figures use: the median forecast is appended to
    the context and the model re-invoked until `gen_len` samples exist. It is imposed from outside
    and is not Chronos-Bolt-Tiny's own generation mode, which emits its whole horizon in one step; it is
    used here because 64 samples give 8 Hz bins, too coarse to place a line.
    """
    ctx_len = len(context)
    ctx = np.asarray(context, dtype=np.float32)[None, :]
    if mode == "horizon":
        return probe.forecast(ctx)[0].astype(float)
    gen = np.zeros((1, 0), dtype=np.float32)
    while gen.shape[1] < gen_len:
        step = probe.forecast(ctx)
        gen = np.concatenate([gen, step], axis=1)
        ctx = np.concatenate([ctx, step], axis=1)[:, -ctx_len:]
    return gen[0, :gen_len].astype(float)


class PublishedProbe:
    """The original Chronos-Bolt-Tiny checkpoint, wrapped in the little of `pl.Probe` used here.

    `pl.Probe` resolves a geometry to one of the project's retrained checkpoints, so it cannot
    load the released model. Only `forecast` is needed, and the geometry is read back off the
    loaded config rather than assumed, so the lock markers are always the model's own.
    """
    def __init__(self, hf_id=PUBLISHED_ID, device=None, batch_size=64):
        import torch
        from chronos import BaseChronosPipeline
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.pipe = BaseChronosPipeline.from_pretrained(hf_id, revision=comparison.OFFICIAL_REVISION, device_map=self.device)
        cfg = self.pipe.model.config.chronos_config
        self.P = int(cfg["input_patch_size"])
        self.S = int(cfg["input_patch_stride"])
        quantiles = list(cfg["quantiles"])
        self.qi = quantiles.index(0.5) if 0.5 in quantiles else len(quantiles) // 2
        self.tag = f"{pl.model_tag(self.P, self.S)}-published"
        self.label = f"{hf_id}  (P={self.P}, S={self.S})"

    def forecast(self, contexts):
        torch = self.torch
        X = np.asarray(contexts, np.float32)
        if X.ndim == 1:
            X = X[None, :]
        out = []
        with torch.no_grad():
            for i in range(0, len(X), self.batch_size):
                xb = torch.tensor(X[i:i + self.batch_size], device=self.device)
                out.append(self.pipe.predict(xb, prediction_length=PRED)[:, self.qi, :]
                           .float().cpu().numpy())
        return np.concatenate(out, axis=0)

    def close(self):
        del self.pipe
        self.pipe = None
        if self.device == "cuda":
            self.torch.cuda.empty_cache()


def open_model():
    """The model this run uses: the stub, the released checkpoint, or the retrained geometry."""
    if USE_STUB_FORECASTER:
        return StubProbe(REF_P, REF_S)
    if MODEL_SOURCE == "published":
        return PublishedProbe(PUBLISHED_ID, batch_size=BATCH)
    if MODEL_SOURCE != "retrained":
        raise ValueError(f"MODEL_SOURCE must be 'retrained' or 'published', not {MODEL_SOURCE!r}")
    return pl.Probe(REF_P, REF_S, batch_size=BATCH)


def real_segment(kind=None, n=None):
    """The stretch of the real signal put beside the prediction, and where it starts.

    The prediction covers `n` samples immediately after the context, so the continuation over
    exactly those instants is the comparison of record: same instants, same length, same
    frequency resolution. The other two settings are there for inspection.
    """
    kind = REAL_SEGMENT if kind is None else kind
    n = len(OUTPUT) if n is None else int(n)
    if kind == "future":
        return np.asarray(SIGNAL[CTX:CTX + n], float), CTX
    if kind == "context_tail":
        return np.asarray(SIGNAL[CTX - n:CTX], float), CTX - n
    if kind == "full":
        return np.asarray(SIGNAL, float), 0
    raise ValueError(f"REAL_SEGMENT must be 'future', 'context_tail' or 'full', not {kind!r}")


def build_output():
    """Load the model, run it on the current context, and bind the prediction beside its real twin.

    A function for the same reason section 1 is one: the second family of section 6 needs the
    whole of it again, and a copy of these lines would be a second place for them to drift.
    """
    global OUTPUT, OFFICIAL_OUTPUT, REAL, REAL_T0, REAL_SPECTRAL_PEAKS, MODEL_LABEL, REF_P, REF_S

    probe = open_model()
    try:
        print("model:", probe.label)
        if INJECTED_LOCK is not None and (probe.P, probe.S) != (REF_P, REF_S):
            print(f"WARNING: the lock was planted for p{REF_P}-s{REF_S} but the checkpoint is "
                  f"p{probe.P}-s{probe.S}; {INJECTED_LOCK:g} Hz may not be a lock of it")
        REF_P, REF_S = probe.P, probe.S      # the loaded geometry decides where the locks are drawn
        MODEL_LABEL = ("stub forecaster, not a model" if USE_STUB_FORECASTER else
                       f"{PUBLISHED_ID}, P={REF_P}, S={REF_S}" if MODEL_SOURCE == "published"
                       else f"retrained p{REF_P}-s{REF_S}")
        print(f"    every file written from here on ends in _seed{ACTIVE_SEED}.*")
        if USE_STUB_FORECASTER:
            print("    NOTE: the stub forecaster is in use, so the name says 'stub' and not "
                  f"'{MODEL_SOURCE}'; these pages show the layout, not the model")
        # A rollout is pointless when only the first PRED samples are kept: the leading PRED samples
        # of a rollout are the one-shot forecast, so the loop would be thrown away.
        mode = "horizon" if (PRED_CUT is not None and PRED_CUT <= PRED) else OUT_MODE
        if mode != OUT_MODE:
            print(f"PRED_CUT={PRED_CUT} <= horizon {PRED}: taking the single forecast, no rollout")
        OUTPUT = chronos_generate(probe, CONTEXT, mode=mode)
    finally:
        probe.close()

    if PRED_CUT is not None:
        OUTPUT = OUTPUT[:int(PRED_CUT)]
    if MODEL_SOURCE == "published":
        OFFICIAL_OUTPUT = OUTPUT  # already official: no second copy or second inference
    else:
        official = (StubProbe(16, 16) if USE_STUB_FORECASTER else
                    comparison.Forecaster(comparison.MODELS[0], batch_size=BATCH))
        try:
            OFFICIAL_OUTPUT = chronos_generate(official, CONTEXT, mode=mode)
        finally:
            official.close()
        if PRED_CUT is not None:
            OFFICIAL_OUTPUT = OFFICIAL_OUTPUT[:int(PRED_CUT)]
    REAL, REAL_T0 = real_segment()
    # These are the peaks of the exact real segment plotted in Figure 4. KernelSynth's
    # REF_FREQS were read earlier from the full 544-sample draw and cannot mark this shorter
    # spectrum; for TSM, REF_FREQS are generator components rather than detected maxima.
    REAL_SPECTRAL_PEAKS = top_peaks(REAL, n_peaks=N_PEAKS_REAL)

    print(f"real segment: '{REAL_SEGMENT}', {len(REAL)} samples starting at "
          f"{REAL_T0 / FS * 1000:.0f} ms")
    print("real spectral peaks [Hz]:", [round(f, 2) for f in REAL_SPECTRAL_PEAKS])
    print(f"context {len(CONTEXT)} samples ({len(CONTEXT) / FS * 1000:.0f} ms), "
          f"prediction {len(OUTPUT)} samples ({len(OUTPUT) / FS * 1000:.0f} ms), "
          f"std {OUTPUT.std():.4f}")
    print(f"spectra evaluated on a {SPECTRUM_BIN_HZ:g} Hz grid (zero padded)")
    print(f"resolving power: prediction {FS / len(OUTPUT):.1f} Hz, real segment {FS / len(REAL):.1f} Hz"
          f" -- two lines closer than that are one line, not two")
    return OUTPUT


build_output()

## 3, Decomposition: fitting the components

In [ ]:
C_IN, C_OUT = "#1f4e79", "#c81e3c"
C_PLANT = "#7d3c98"                         # the planted component, wherever it turns up
LOCKS_REF = locks_of(REF_P, REF_S)          # the loaded geometry, so the markers are its own


def frequency_label(f):
    """Display ratios from Section 2, using the loaded model geometry."""
    return f"{f * REF_P / FS:.3g} cpp / {f * REF_S / FS:.3g} cps"


def is_planted(f, tol=1e-9):
    """Whether `f` is the component planted on a lock frequency before the model saw the signal.

    On the real side the match is exact: that frequency is the one that was added. On the
    prediction side one bin is passed in as the tolerance, because the transform cannot place a
    line more finely than that, so a returned line within a bin of the planted site is that site.
    """
    return INJECTED_LOCK is not None and abs(f - INJECTED_LOCK) <= tol


def report_written(p):
    """Name, size and time of every file written, and the directory it went to.

    A page is read back, or copied to Drive, by name, and a name says nothing about when it was
    made: a run that stops halfway leaves the previous file in place and it looks exactly like a
    fresh one. The time stamp printed here is what tells the two apart.
    """
    import time
    st = p.stat()
    print(f"    wrote {p.name}  {st.st_size / 1024:.0f} kB  "
          f"{time.strftime('%H:%M:%S', time.localtime(st.st_mtime))}  -> {p.parent}")


def component_at(x, f, fs=FS, n=None):
    """Least-squares amplitude and phase of `x` at `f` Hz, the estimator R is fitted with.

    `n` truncates the fit to the leading `n` samples. A window shorter than a cycle of `f` cannot
    separate amplitude from phase, so `cycles_in` is reported beside the fit and flagged on the
    panel when it falls below `MIN_CYCLES_WARN`.
    """
    y = np.asarray(x, float)
    if n is not None:
        y = y[:int(min(int(n), len(y)))]
    t = np.arange(len(y)) / fs
    amp, ph = pl.fit_amp_phase(y, t, f)
    return float(amp), float(ph)


def components_of(x, freqs, n=None, joint=None, fs=FS):
    """Amplitude and phase of `x` at each of `freqs`, as a list of (amp, phase).

    With `joint`, one design matrix carries every frequency at once, so energy shared by two
    frequencies closer than the window can resolve is split between them instead of being
    reported in full for each. An ill-conditioned design means exactly that: the window cannot
    tell those frequencies apart, and the notebook says so rather than returning a tidy number.
    """
    joint = JOINT_FIT if joint is None else joint
    y = np.asarray(x, float)
    if n is not None:
        y = y[:int(min(int(n), len(y)))]
    t = np.arange(len(y)) / fs
    if not joint or len(freqs) < 2:
        return [component_at(y, f, fs=fs) for f in freqs]
    cols = []
    for f in freqs:
        cols += [np.cos(2 * np.pi * f * t), np.sin(2 * np.pi * f * t)]
    X = np.stack(cols + [np.ones_like(t)], axis=1)
    cond = np.linalg.cond(X)
    if cond > 1e3:
        print(f"    joint fit poorly conditioned (cond = {cond:.0f}): "
              f"{FS / len(y):.1f} Hz bins cannot separate these frequencies")
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return [(float(np.hypot(beta[2 * i], beta[2 * i + 1])),
             float(np.arctan2(beta[2 * i + 1], beta[2 * i]))) for i in range(len(freqs))]


def spectrum_magnitudes_at(x, freqs, fs=FS):
    """The plotted Hann-spectrum magnitude at each requested frequency."""
    fr, mag = spectrum(x, fs=fs)
    return [float(mag[int(np.argmin(np.abs(fr - f)))]) for f in freqs]


def prediction_selection(n_peaks=None, min_magnitude=None, verbose=True, signal=None):
    """Select prediction frequencies in the same magnitude domain Figure 4 displays.

    `top_peaks` supplies a permissive candidate set. The final cut is applied directly to the
    Hann-spectrum magnitudes plotted in Figure 4, relative to the strongest candidate. Figure 3
    receives this exact frequency list and uses joint least squares only to estimate the retained
    components' amplitudes and phases; the fit never changes membership.
    """
    signal = OUTPUT if signal is None else signal
    candidates = top_peaks(signal, n_peaks=n_peaks)
    magnitudes = spectrum_magnitudes_at(signal, candidates)
    strongest = max(magnitudes, default=0.0)
    if min_magnitude is not None:
        floor, rule = float(min_magnitude), "explicit absolute floor"
    elif SPECTRAL_CUT_ABS is not None:
        floor, rule = float(SPECTRAL_CUT_ABS), "configured absolute floor"
    else:
        floor = SPECTRAL_CUT_REL * strongest
        rule = f"{100 * SPECTRAL_CUT_REL:g}% of the strongest plotted prediction peak"
    kept = [f for f, magnitude in zip(candidates, magnitudes) if magnitude >= floor]
    magnitude_by_frequency = {float(f): float(magnitude)
                              for f, magnitude in zip(candidates, magnitudes)}
    selection = dict(candidates=list(candidates), frequencies=kept,
                     magnitude_by_frequency=magnitude_by_frequency,
                     floor=float(floor), strongest=float(strongest), rule=rule)
    if verbose:
        dropped = len(candidates) - len(kept)
        if dropped:
            print(f"    {dropped} prediction line(s) below spectral M = {floor:.4f} "
                  f"({rule}) dropped")
        kept_text = ", ".join(
            f"{f:.2f} Hz (M={magnitude_by_frequency[f]:.3f})" for f in kept)
        print(f"    kept: {kept_text or 'none'}")
    return selection


def resolution_hz(x, fs=FS):
    """Width of one bin: two frequencies closer than this are one line, not two."""
    return fs / len(x)


def near_input(f, tol=None):
    """The plotted real component `f` is within one bin of, if any."""
    tol = resolution_hz(OUTPUT) if tol is None else tol
    reference = REF_FREQS if FAMILY == "tsmixup" else REAL_SPECTRAL_PEAKS
    if not len(reference):
        return None
    j = int(np.argmin([abs(f - g) for g in reference]))
    return reference[j] if abs(f - reference[j]) <= tol else None


def cycles_in(f, n, fs=FS, total=None):
    """How many cycles of `f` a fit window of `n` samples spans."""
    n = total if n is None else min(int(n), total or int(n))
    return float("inf") if n is None else f * n / fs


def window_note(n):
    return "the whole trace" if n is None else f"the first {int(n)} samples"


def component_curve(f, amp, ph, n_cycles=None, n_pts=800, span=None, t0_samples=0):
    """The fitted sinusoid on a fine grid, so a 200 Hz component is not drawn from eight samples.

    `span` is a length in samples: the component is then drawn over the same stretch of absolute
    time as the signal it was fitted on, which is what puts every panel of the page on one axis.
    """
    if COMPONENT_WINDOW == "signal" and span:
        t = np.linspace(0, span / FS, n_pts)
    else:
        n_cycles = COMPONENT_CYCLES if n_cycles is None else n_cycles
        t = np.linspace(0, n_cycles / max(f, 1e-9), n_pts)
    y = amp * np.cos(2 * np.pi * f * t - ph)
    return (t + t0_samples / FS) * 1000.0, y


def params_caption(width=118):
    """Three short lines: what generated the input, what it is made of, and what predicted it."""
    import textwrap
    if FAMILY == "tsmixup":
        pool_now = source_pool(verbose=False)
        band_txt = f"{frequency_label(min(pool_now))} to {frequency_label(max(pool_now))}"
        drawn = [f for f in REF_FREQS if f != INJECTED_LOCK]
        w_of = dict(zip(sorted(drawn), [WEIGHTS[i] for i in np.argsort(np.argsort(drawn))]))
        head = (f"Input : Light TSMixup, K={len(drawn)}, alpha={ALPHA_DIR}, "
                f"drawn from {band_txt}, seed {ACTIVE_SEED}"
                + (f", planted lock at {frequency_label(INJECTED_LOCK)} "
                   f"({INJECT_LOCK_REL:g}x)" if INJECTED_LOCK else ""))
        body = " | ".join(
            f"{frequency_label(f)} " + (f"(w={w_of[f]:.2f})" if f in w_of else "(planted)")
            + (" [lock]" if is_lock(f) else "") for f in REF_FREQS)
    else:
        head = (f"Input : KernelSynth, J={KS_PARAMS.get('J')}, l_syn={KS_PARAMS.get('l_syn')}, "
                f"jitter={KS_PARAMS.get('jitter')}, seed {ACTIVE_SEED}"
                + (f", planted lock at {frequency_label(INJECTED_LOCK)}" if INJECTED_LOCK else ""))
        body = "kernels: " + " ".join(KERNELS)
    model = (f"Model : {MODEL_LABEL}, prediction {len(OUTPUT)} samples "
             f"({frequency_label(FS / len(OUTPUT))} bins), fit on {window_note(FIT_WINDOW_PRED)}")
    body = textwrap.fill(body, width=width, subsequent_indent="        ")
    return f"{head}\n        {body}\n{model}"


def add_params_strip(fig, y=0.012):
    """The draw's own parameters, printed on the figure so it can be read without the notebook."""
    fig.text(0.045, y, params_caption(), ha="left", va="bottom", fontsize=7.0, color="0.3",
             family="monospace", linespacing=1.5)


def draw_signal(ax, y, color, title, fs=FS, t0_samples=0):
    """One signal against absolute time, so a panel's width means the same thing on both sides."""
    t = (np.arange(len(y)) + t0_samples) / fs * 1000.0
    ax.plot(t, y, color=color, lw=0.8)
    ax.set_title(f"{title}\n{len(y)} samples, {len(y) / fs * 1000:.0f} ms", fontsize=9, pad=4)
    ax.set_xlabel("time [ms]", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.margins(x=0.01)


def panel_widths(base_l=1.45, base_r=1.45, floor=0.32):
    """Widths for the two signal columns: in proportion to how much time each one covers.

    The prediction is a horizon, not a second recording: it spans a fraction of the context, and
    equal-width panels would silently stretch it to look as long. The floor keeps a very short
    horizon legible.
    """
    if not MATCH_TIME_SCALE:
        return base_l, base_r
    ratio = float(np.clip(len(OUTPUT) / max(len(REAL), 1), floor, 1.0))
    return base_l, base_r * ratio


def page_ylim(amps=(), reference_frequencies=None):
    """The half-height every component panel of a page is drawn to.

    One scale for the page, set by the strongest component of the real signal: autoscaling each
    panel makes a component at one per cent of the input fill its box and read as a line.
    """
    reference_frequencies = (REF_FREQS if reference_frequencies is None
                             else reference_frequencies)
    ref = max([a for a, _ in components_of(REAL, reference_frequencies, n=FIT_WINDOW_REAL)]
              + [1e-9])
    if AMP_SCALE == "own":
        return 1.15 * max(list(amps) + [1e-9])
    return 1.15 * max([ref] + list(amps))       # never clip a component that exceeds the scale


def amp_ticks(ax, ylim, frac=1.15, mark=None):
    """Ticks for the panel's scale, plus the amplitude it actually holds when the two differ.

    With one scale shared across a page every panel would otherwise carry the same three numbers,
    which says what the axis is but not what the component is.
    """
    a = ylim / frac
    ticks = [-a, 0.0, a]
    if mark and 0.12 * a < mark < 0.93 * a:
        ticks += [-mark, mark]
    ticks = sorted(set(round(t, 6) for t in ticks))
    ax.set_yticks(ticks)
    ax.set_yticklabels(["0" if abs(t) < 1e-9 else f"{t:.3g}" for t in ticks], fontsize=5)
    if mark and 0.12 * a < mark < 0.93 * a:
        for v in (-mark, mark):
            ax.axhline(v, color="0.6", lw=0.5, ls=":", zorder=1)
    ax.set_ylabel("amplitude", fontsize=5.5)


def arrow(fig, ax_from, ax_to, side="right"):
    """An arrow from the edge of a signal panel to the edge of a component panel."""
    if side == "right":
        a, b = (1.005, 0.5), (-0.05, 0.5)
    else:
        a, b = (-0.005, 0.5), (1.05, 0.5)
    fig.add_artist(ConnectionPatch(xyA=a, coordsA=ax_from.transAxes,
                                   xyB=b, coordsB=ax_to.transAxes,
                                   arrowstyle="-|>", mutation_scale=11, lw=0.8, color="0.45"))

## 4. Real, retrained and official components

Each column uses its own selected frequencies and states its checkpoint and geometry.
TSMixup's real frequencies come from its draw; KernelSynth uses the local spectral maxima
of the actual displayed real segment. Component fits do not change peak membership.
The same selected output peaks are reused in the spectrum figure below.


In [ ]:
def comparison_groups(selection=None):
    selection = prediction_selection() if selection is None else selection
    real_freqs = list(REF_FREQS if FAMILY == "tsmixup" else REAL_SPECTRAL_PEAKS)
    groups = [dict(side="real", label=f"{NAME} real ({REAL_SEGMENT}); reference geometry",
                   values=REAL, t0=REAL_T0, P=REF_P, S=REF_S, frequencies=real_freqs,
                   spectrum_peaks=list(REAL_SPECTRAL_PEAKS))]
    selected_side = "official" if MODEL_SOURCE == "published" else "retrained"
    groups.append(dict(side=selected_side, label=MODEL_LABEL, values=OUTPUT, t0=CTX,
                       P=REF_P, S=REF_S, frequencies=list(selection["frequencies"]),
                       selection=selection))
    if MODEL_SOURCE != "published":
        official_selection = prediction_selection(signal=OFFICIAL_OUTPUT, verbose=False)
        groups.append(dict(side="official", label="amazon/chronos-bolt-tiny official",
                           values=OFFICIAL_OUTPUT, t0=CTX, P=16, S=16,
                           frequencies=list(official_selection["frequencies"]),
                           selection=official_selection))
    return groups


def own_components_page(fname, selection=None):
    return comparison_plot.components_page(
        comparison_groups(selection), OUT / "figures" / fname,
        title=f"{NAME}: real and checkpoint components | seed {ACTIVE_SEED}",
        fit_real=FIT_WINDOW_REAL, fit_pred=FIT_WINDOW_PRED, joint=JOINT_FIT,
        pdf=pdf_requested(), stub=USE_STUB_FORECASTER, planted=INJECTED_LOCK,
        component_window=COMPONENT_WINDOW, component_cycles=COMPONENT_CYCLES,
        amp_scale=AMP_SCALE, match_time_scale=MATCH_TIME_SCALE)


PREDICTION_SELECTION = prediction_selection()
page_c, table_c = own_components_page(
    seeded_name(f"FIG_E3_{run_suffix()}_three_way_components", ".png"), PREDICTION_SELECTION)
table_c.to_csv(OUT / seeded_name(f"components_{run_suffix()}", ".csv"), index=False)
from IPython.display import Image, display
display(Image(filename=str(page_c), width=1200))
display(table_c.round(4))


## 5. Three spectra on the same input

The real segment and both predictions are shown in separate panels. Each panel's cpp/cps
coordinates and lock markers use its declared patch/stride. The prediction threshold is
computed independently for each checkpoint; black triangles mark exactly the components
retained in the preceding page. The injected-frequency matching tolerance is ±1 Hz.


In [ ]:
def spectra_page(fname, selection=None):
    return comparison_plot.spectra_page(
        comparison_groups(selection), OUT / "figures" / fname,
        title=f"{NAME}: real and checkpoint spectra | seed {ACTIVE_SEED}",
        drawn=([f for f in REF_FREQS if not is_planted(f)] if FAMILY == "tsmixup" else None),
        planted=INJECTED_LOCK, pdf=pdf_requested(), stub=USE_STUB_FORECASTER,
        bin_hz=SPECTRUM_BIN_HZ)


page_d = spectra_page(
    seeded_name(f"FIG_E4_{run_suffix()}_three_way_spectra", ".png"), PREDICTION_SELECTION)
display(Image(filename=str(page_d), width=1000))


## 6, Every requested seed and family, on the same model

Figures 3 and 4 are wanted for both generators. `DRAW_SEED` may be one integer or an iterable;
this section completes the family-by-seed matrix after sections 1-5 produced its first member.
Each draw passes through the same checkpoint and the active seed is the final filename token,
so no family or seed overwrites another.

The globals are left holding the last family and seed run. To go back, re-run sections 1 and 2.


In [ ]:
FIGURE_FAILURES = []
RUNS = [(seed, fam) for seed in DRAW_SEEDS for fam in ("tsmixup", "kernelsynth")
        if not (seed == DRAW_SEEDS[0] and fam == PRIMARY_FAMILY)]

for seed, fam in RUNS:
    try:                       # one draw failing must not cost the others their pages
        build_signal(fam, seed)
        describe_draw(show=False)
        build_output()
        _selection = prediction_selection()
        _p3, _t3 = own_components_page(
            seeded_name(f"FIG_E3_{run_suffix()}_main_components", ".png"), _selection)
        _t3.to_csv(OUT / seeded_name(f"components_{run_suffix()}", ".csv"), index=False)
        _p4 = spectra_page(
            seeded_name(f"FIG_E4_{run_suffix()}_spectra", ".png"), _selection)
        print(f"completed {NAME}, seed {ACTIVE_SEED}")
        print("Saved component and spectrum pages:", _p3, _p4)
        display(Image(filename=str(_p3), width=1200))
        display(Image(filename=str(_p4), width=1000))
        display(_t3.round(4))
    except Exception as e:
        FIGURE_FAILURES.append((fam, seed, str(e)))
        print(f"{fam}, seed {seed} failed: {type(e).__name__}: {e}")
        if isinstance(e, ModuleNotFoundError):
            print("   KernelSynth needs chronos/data/synthetic/signalGenerator.py on the path; "
                  "probe_lib adds that directory when it imports, so a checkout without the file "
                  "is the usual cause")

print("globals now hold:", NAME, "| seed", ACTIVE_SEED, "|", run_suffix())
if FIGURE_FAILURES:
    raise RuntimeError(f"Incomplete figure matrix: {FIGURE_FAILURES}")


## 7, The files this run wrote

Every page is written as PNG and, only when `SAVE_PDF = "Y"`, also as PDF. Its name carries
the family, model, planted component when present, and the seed as the final token. What goes
into the report is a row
marked *this run*: files from earlier runs sit in the same folder under different names and are
easy to pick up by mistake.


In [ ]:
import time

_rows = []
for f in sorted((OUT / "figures").glob("*")) + sorted(OUT.glob("*.csv")):
    st = f.stat()
    _rows.append(dict(file=f.name, kB=round(st.st_size / 1024),
                      modified=time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(st.st_mtime)),
                      family=next((k for k in ("tsmixup", "kernelsynth") if k in f.name), ""),
                      model=("published" if "published" in f.name else
                             "stub" if "stub" in f.name else "retrained"),
                      planted=("yes" if "_lock" in f.name else ""),
                      seed=(f.stem.rsplit("_seed", 1)[1] if "_seed" in f.stem else "")))
FILES = pd.DataFrame(_rows).sort_values("modified", ascending=False)
print(f"{OUT}")
print("figures of this session carry the tags:",
      ", ".join(sorted({r["file"].rsplit("_seed", 1)[0].split("FIG_E")[-1][2:].rsplit("_", 2)[0]
                        for r in _rows if r["file"].startswith("FIG_E")})) or "none yet")
display(FILES)

In [ ]:
# Saving the figures somewhere that outlives the runtime.  On Colab `_run/` lives on the VM's own
# disk and disappears when the session ends, so the figures are mirrored to Drive; elsewhere the
# same call copies them to DRIVE_DIR if that path exists, and is otherwise a no-op.
import importlib.util, shutil

DRIVE_COPY = False
DRIVE_DIR  = "/content/drive/MyDrive/appendixE_figures"


def mirror_to_drive(src=None, dest=DRIVE_DIR, patterns=("*.png", "*.pdf", "*.csv")):
    """Copy the figures out of `_run/` and into Drive.  Returns the destination, or None."""
    if not DRIVE_COPY:
        print("DRIVE_COPY is off; figures stay under", OUT)
        return None
    src = Path(src or (OUT / "figures"))
    on_colab = importlib.util.find_spec("google.colab") is not None
    if on_colab and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")                 # asks once per runtime
    dest = Path(dest)
    if not on_colab and not dest.parent.exists():
        print(f"not on Colab and {dest.parent} does not exist; figures stay under {src}")
        return None
    dest.mkdir(parents=True, exist_ok=True)
    copied = []
    for pattern in patterns:
        for f in sorted(list(src.glob(pattern)) + list(src.parent.glob(pattern))):
            shutil.copy2(f, dest / f.name)
            copied.append(f.name)
    print(f"copied {len(copied)} files -> {dest}")
    for name in copied:
        print("   ", name)
    return dest


mirror_to_drive()

## 8. Interpretation

These figures are descriptive comparisons on identical contexts. The official model and
the retrained 16,16 have different training histories. Their difference alone does not
identify an effect of patching. q50 is a median forecast, not a sampled trajectory.
An optional rollout in the figure configuration feeds predictions back into the model;
the recovery tables below always use the native 64-point forecast instead.


## 9. Recovery tables: TSMixup and KernelSynth

For each family, generate 100 unit-standard-deviation backgrounds with the original
decomposition recipe (Light TSMixup K uniformly 1–4, alpha 1.5, minimum separation 10 Hz;
KernelSynth J=5, jitter=1e-4). Each geometry receives 100 paired lock/control trials:
amplitude **1.5 × background standard deviation**, common phase within the pair,
no normalization after injection. Backgrounds are reused across all checkpoints.

Locks are the union of patch and stride sites, balanced over 100 draws; repeated frequencies
are necessary because each geometry has fewer than 100 distinct locks. Controls are placed
near their lock with balanced lower/upper signs, and are strictly more than 2 Hz from every
lock. The two 16,16 checkpoints have identical complete signals.

Use up to 8 separated local Hann-spectrum peaks, then retain peaks at least 35% of the
strongest. Recovery means any retained peak lies within inclusive ±1 Hz of the injection.
All 100 trials remain in each denominator. An invalid prediction interrupts the run rather
than being silently excluded. Percentages measure output spectral presence and do not
subtract the background's pre-existing spectral content.

Full results contain 16 rows per family. Smoke results are stored separately and are not
reportable as the full experiment. Caches record configuration, model revisions, source
fingerprints and exact input identities. Re-running resumes compatible completed work.


In [ ]:
if RUN_RECOVERY_TABLES:
    if USE_STUB_FORECASTER:
        raise ValueError("Recovery tables require real checkpoints; turn off the figure stub")
    RECOVERY_RESULTS = comparison.run_recovery(COMPARISON_ROOT, RECOVERY_CONFIG, COMPARISON_MODELS)
    for family, (table, directory) in RECOVERY_RESULTS.items():
        print(f"{family} | {COMPARISON_MODE.upper()} | {directory}")
        display(table.round(2))
